In [1]:
import spacy
from spacy.tokens import Span
import os
import pandas as pd
import json
import pyab3p
import re

path_to_custom_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_ner')

scispacy_model = spacy.load(path_to_custom_model)

c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [2]:
with open('list_files\\ncbi_ids.json', 'r') as f:
    species_ids = json.load(f)

with open('list_files\\microbes_genus_species.json', 'r') as f:
    all_microbe_names = json.load(f)

tags = pd.read_csv('..\\dissertation DLC content\\manual-corpus-species-1.0\\filtered_tags.tsv', sep='\t')

global_found = 0
global_total = 0

standard_species = ['human', 'yeast', 'mice', 'mouse', 'flies', 'goat', 'humans', 'cattle', 'cow', 'cows', 'sheep', 'rat', 'donkey', 'donkeys', 'pig', 'pigs', 'bananas', 'bovine', 'rice', 'fission yeast', 'rice blast fungus', 'anthrax']

for ind, file in enumerate(os.listdir("..\\dissertation DLC content\\manual-corpus-species-1.0\\txt")):
    filename = os.fsdecode(file)[:-4]
    print(filename)
    ab3p_mapping = {}
    with open(f'..\\dissertation DLC content\\manual-corpus-species-1.0\\txt\\{filename}.txt') as f:
        text = f.read()

        filerows = tags.loc[tags['document'] == filename, ['#entity id', 'start', 'end', 'text']]
        #filerows.reset_index(drop=True)

        abbrev = pyab3p.Ab3p()

        abbreviations = abbrev.get_abbrs(text)

        for abbrev in abbreviations:
            abbrev_ents = scispacy_model(abbrev.long_form)
            if 'MICROBE_NAME' in [ent.label_ for ent in abbrev_ents.ents]:
                ab3p_mapping[abbrev.short_form] = abbrev.long_form

        microbes = scispacy_model(text)

        for short, long in ab3p_mapping.items():
            matches = re.finditer(short, text)
            for match in matches:
                span = microbes.char_span(match.start(), match.end(), label="MICROBE_NAME")
                if span != None:
                    #print(span.start, span.end)
                    microbes.set_ents([Span(microbes, span.start, span.end, label='MICROBE_NAME')], default='unmodified')

        found_count = 0
        total_count = 0
        false_positives = 0

        false_positive_names = []

        all_microbe_names = []

        for ind, row in filerows.iterrows():
            try:
                species_id = filerows['#entity id'][ind].split(':ncbi:')[1]
            except:
                continue
            if species_id in species_ids:
                name = row['text'].lower()
                if not name in standard_species:
                    all_microbe_names.append(row['text'])
                    total_count += 1

        names_found = []

        for ent in microbes.ents:
            if not 'MICROBE' in ent.label_:
                continue
            if (ent.text not in all_microbe_names) and (ent.text not in list(ab3p_mapping.keys())):
                continue
            """ try:
                print(filerows['#entity id'])
                species_id = filerows['#entity id'][1].split(':ncbi:')[1]
            except:
                continue
            if not species_id in species_ids:
                continue """
            row_found = filerows[(filerows['text'] == ent.text) & (filerows['start'] == ent.start_char) & (filerows['end'] == ent.end_char)]
            if not row_found.empty:
                found_count += 1
                names_found.append(ent.text)
            else:
                false_positives += 1
                false_positive_names.append(ent.text)

        global_found += found_count
        global_total += total_count

        print(f"found {found_count} / {total_count}")
        print(f"false positives: {false_positives}")
        print(f"microbes identified: {names_found}")
        print()
        print(f"false positive microbes: {false_positive_names}")
        print()
        print(f"all microbes: {all_microbe_names}")
        print()

print(f"total found: {global_found} / {global_total}")

pmcA102792
found 9 / 9
false positives: 0
microbes identified: ['Saccharomyces cerevisiae', 'Saccharomyces cerevisiae', 'Escherichia coli', 'S.cerevisiae', 'Schizosaccharomyces pombe', 'S.cerevisiae', 'E.coli', 'Pichia pastoris', 'S.cerevisiae']

false positive microbes: []

all microbes: ['Saccharomyces cerevisiae', 'Saccharomyces cerevisiae', 'Escherichia coli', 'S.cerevisiae', 'Schizosaccharomyces pombe', 'S.cerevisiae', 'E.coli', 'Pichia pastoris', 'S.cerevisiae']

pmcA1036069
found 0 / 0
false positives: 0
microbes identified: []

false positive microbes: []

all microbes: []

pmcA1075922
found 0 / 0
false positives: 0
microbes identified: []

false positive microbes: []

all microbes: []

pmcA1079799
found 0 / 0
false positives: 0
microbes identified: []

false positive microbes: []

all microbes: []

pmcA1131934
found 0 / 0
false positives: 0
microbes identified: []

false positive microbes: []

all microbes: []

pmcA1190194
found 1 / 2
false positives: 0
microbes identified: ['